# Support Vector Machine with RBF kernel

## Goal of this notebook
The logistic regression notebook showed that a linear model **underfits** (train and validation macro F1 both around 0.55). This notebook tests whether a **non-linear** decision boundary, obtained with an SVM and an RBF kernel, does better.

## Decisions taken (same as the logistic regression notebook)
- 5 AAMI classes.
- Scores: macro F1 (used for model selection) and macro recall.
- Preprocessing: the PCA analysis and the comparison of the preprocessing methods were done in `logreg_pca_preprocessing.ipynb` and are not repeated here.

## Notebook chapters
| Section | What we do |
|---|---|
| 1 | Load the data and remap to the 5 AAMI classes |
| 2 | Evaluation tools |
| 3 | Why an SVM, and why the RBF kernel |
| 4 | Preprocessing: P1 + PCA(100) |
| 5 | Computational cost: tuning on a subset of the training set |
| 6 | Grid search over C and gamma |
| 7 | Final model trained on the full training set |
| 8 | Test set |
| 9 | Conclusions |

## 0. Imports and constants

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data_preprocessing.py").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project folder or a notebook subfolder.")

DATA_DIR = PROJECT_ROOT / "data"
if not all((DATA_DIR / name).is_file() for name in (
    "train_data.csv", "val_data.csv", "test_data.csv"
)):
    raise FileNotFoundError("Dataset missing: see data/README.md for the required CSV files.")


# Names of the 15 original labels (index = numeric label in the csv files)
ORIGINAL_NAMES = ["N", "/", "L", "R", "e", "j", "A", "a", "J", "S", "E", "F", "V", "f", "Q"]

# Names of the 5 AAMI classes (index = new numeric label we will use)
AAMI_NAMES = ["N", "S", "V", "F", "Q"]
N_CLASSES = len(AAMI_NAMES)

## 1. Load the data and remap the labels

Same functions as in the logistic regression notebook (see Sections 1 and 2 there for the explanation).

In [ ]:
# Load one split (train, validation or test) from a csv file.
# The labels are the original ones, 0 to 14.

def load_split(path):
    data = pd.read_csv(path, header=None)

    X = data.iloc[:, :-1].to_numpy(dtype=float) 
    y = data.iloc[:, -1].to_numpy().astype(int) 

    return X, y

In [ ]:
X_train, y_train_orig = load_split(DATA_DIR / "train_data.csv")
X_val, y_val_orig = load_split(DATA_DIR / "val_data.csv")
X_test, y_test_orig = load_split(DATA_DIR / "test_data.csv")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train_orig.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val_orig.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test_orig.shape)

In [ ]:
# Original numeric label -> AAMI numeric label
ORIGINAL_TO_AAMI = {
    0: 0,    # N -> N
    1: 4,    # / -> Q
    2: 0,    # L -> N
    3: 0,    # R -> N
    4: 0,    # e -> N
    5: 0,    # j -> N
    6: 1,    # A -> S
    7: 1,    # a -> S
    8: 1,    # J -> S
    9: 1,    # S -> S
    10: 2,   # E -> V
    11: 3,   # F -> F
    12: 2,   # V -> V
    13: 4,   # f -> Q
    14: 4,   # Q -> Q
}

# Convert an array of original labels (0..14) into AAMI labels (0..4).
def to_aami(y_orig):
    
    lookup = np.array([ORIGINAL_TO_AAMI[i] for i in range(len(ORIGINAL_NAMES))])

    return lookup[y_orig]

In [ ]:
y_train = to_aami(y_train_orig)
y_val = to_aami(y_val_orig)
y_test = to_aami(y_test_orig)

print("y_train labels: ",np.bincount(y_train, minlength=N_CLASSES))
print("y_val labels: ", np.bincount(y_val, minlength=N_CLASSES))   
print("y_test labels: ", np.bincount(y_test, minlength=N_CLASSES))

## 2. Evaluation tools

Same `evaluate`, `plot_confusion_matrix` and `fit_and_score` as in the logistic regression notebook, so the numbers of the two notebooks can be compared directly.

In [ ]:
# Compute the scores we use to compare models.
def evaluate(y_true, y_pred):
    labels = list(range(N_CLASSES))

    recall_per_class = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    f1_per_class = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)

    # Macro = plain mean over the 5 classes
    scores = {
        "f1_macro": f1_per_class.mean(),
        "recall_macro": recall_per_class.mean(),
    }
    for i in range(N_CLASSES):
        name = AAMI_NAMES[i]
        scores["recall_" + name] = recall_per_class[i]
        scores["f1_" + name] = f1_per_class[i]

    return scores

# Plot the confusion matrix.
# normalize=True: each row is divided by its sum, so the diagonal is the per-class recall
# (needed because some classes are very rare).
def plot_confusion_matrix(y_true, y_pred, title="", normalize=True):
    conf_mat = confusion_matrix(y_true, y_pred, labels=range(N_CLASSES), normalize="true" if normalize else None)
    conf_disp = ConfusionMatrixDisplay(conf_mat, display_labels=AAMI_NAMES)
    conf_disp.plot(cmap=plt.cm.Blues, values_format=".2f" if normalize else "d")
    conf_disp.ax_.set_title(title)
    plt.show()

# Fit the model on the training data and evaluate it on train and validation.
def fit_and_score(model, X_tr, y_tr, X_va, y_va):

    model.fit(X_tr, y_tr)

    scores_train = evaluate(y_tr, model.predict(X_tr))
    scores_val = evaluate(y_va, model.predict(X_va))

    result = {}
    for key in scores_train:
        result["train_" + key] = scores_train[key]
    for key in scores_val:
        result["val_" + key] = scores_val[key]

    return result

## 3. Why an SVM, and why the RBF kernel

In this notebook we will not be training a linear SVM. A linear SVM separates the classes with hyperplanes, like logistic regression; only the loss changes (hinge instead of cross-entropy). Since the linear model underfits, a linear SVM is expected to hit the same limit.

Using the RBF kernel gives us a non-linear decision boundary, because a new beat is classified with a weighted sum of its similarities to the *support vectors*.
$$k(x, x') = \exp(-\gamma \|x - x'\|^2)$$

**Hyperparameters**
- **`C`**: cost of a margin violation. Large C -> few training errors allowed, more complex boundary (risk of overfitting). Small C -> wider margin, smoother boundary.
- **`gamma`**: inverse width of the kernel. Large gamma -> each support vector only influences its close neighbours, very wiggly boundary (overfitting). Small gamma -> smooth boundary, close to a linear one (underfitting).

**Other settings**
- `class_weight="balanced"`, as for logistic regression: errors on the rare classes cost more.
- Multiclass: `SVC` trains one binary SVM for every pair of classes (one-vs-one, 10 SVMs for 5 classes) and takes a vote.

In [ ]:
# return an unfitted SVM with RBF kernel
def build_svm(C, gamma):
    model = SVC(
        kernel="rbf",
        C=C,
        gamma=gamma,
        class_weight="balanced",
        cache_size=1000,  # MB of memory for the kernel values, speeds up the training
    )
    return model

## 4. Preprocessing: P1 + PCA(100)

- **P1 (baseline removal + amplitude normalisation)** was the best preprocessing for logistic regression, so we keep it.

- **PCA with k = 100**: PCA without whitening is a rotation of the data followed by dropping the last components. With k = 100 the reconstruction error is almost the same for all the classes (logreg notebook, Section 5.4). The gain is speed: every kernel value is computed on 100 numbers instead of 250.

In [ ]:
# Preprocessing: remove baseline and normalize amplitude of each beat (row).
def remove_baseline(X):

    baseline = np.median(X, axis=1, keepdims=True)   

    X_centered = X - baseline
    return X_centered


def normalize_amplitude(X, eps=1e-8):

    scale = np.std(X, axis=1, keepdims=True)

    X_scaled = X / (scale + eps)
    return X_scaled

In [ ]:
K_PCA = 100

# P1: per-beat operations, nothing to fit
X_train_p1 = normalize_amplitude(remove_baseline(X_train))
X_val_p1 = normalize_amplitude(remove_baseline(X_val))
X_test_p1 = normalize_amplitude(remove_baseline(X_test))

# PCA fitted on train only
pca = PCA(n_components=K_PCA, random_state=RANDOM_STATE)
pca.fit(X_train_p1)
X_train_svm = pca.transform(X_train_p1)
X_val_svm = pca.transform(X_val_p1)
X_test_svm = pca.transform(X_test_p1)

# To check if the analysis done in logreg notebook holds
print(f"Variance kept by {K_PCA} components (after P1): {pca.explained_variance_ratio_.sum():.4f}")

## 5. Computational cost: tuning on a subset

Training a kernel SVM on all 65 670 beats for each of the 20 combinations of the grid would cost too much time, so we reduce the data used for tuning.

What we do to avoid this is:
1. Tune `C` and `gamma` on a stratified 20% subset of the training set (same class proportions as the full set).
2. Score every combination on the full validation set.
3. Retrain only the best combination on the full training set (Section 7).

Limitation: the best `C` we find using 20% of the training set may not be the best one on the full data.

In [ ]:
# stratify=y_train keeps the class proportions; the other 80% of the split is not used
X_tune, X_unused, y_tune, y_unused = train_test_split(
    X_train_svm, y_train, train_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)

print("Tuning subset:", X_tune.shape)
print("Beats per class:", np.bincount(y_tune, minlength=N_CLASSES))

## 6. Grid search over C and gamma

- **`C` grid:** log-spaced, 0.1 to 100.
- **`gamma` grid:** a good value of gamma depends on the scale of the data, so we use as reference the sklearn default `gamma="scale"`.
- **Selection:** highest validation macro F1.

In [ ]:
gamma_ref = 1 / (X_tune.shape[1] * X_tune.var())
print(f"gamma_ref = {gamma_ref:.4g}")

C_GRID = [0.1, 1, 10, 100]
GAMMA_FACTORS = [0.1, 0.3, 1, 3, 10]

In [ ]:
rows = []

for C in C_GRID:
    for factor in GAMMA_FACTORS:
        gamma = factor * gamma_ref

        start = time.time()
        result = fit_and_score(build_svm(C, gamma), X_tune, y_tune, X_val_svm, y_val)
        elapsed = time.time() - start

        row = {"C": C, "gamma_factor": factor, "gamma": gamma, "time_s": elapsed}
        for key in result:
            row[key] = result[key]
        rows.append(row)

        print(f"C={C:<5g} gamma={factor:g}*ref  val_f1_macro={result['val_f1_macro']:.3f}  ({elapsed:.0f} s)")

svm_results = pd.DataFrame(rows)

### 6.1 Heatmaps: train vs validation

One cell per (C, gamma) combination. Comparing the two heatmaps shows where the model overfits: train F1 close to 1 and validation F1 much lower (expected for large gamma and large C).

In [ ]:
# Plot one score of the grid as a heatmap
def plot_grid_heatmap(results, score, title):
    table = results.pivot(index="C", columns="gamma_factor", values=score)
    values = table.to_numpy()

    plt.figure(figsize=(6, 4))
    plt.imshow(values, cmap="Blues", vmin=0.5, vmax=1)
    plt.colorbar()

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            plt.text(j, i, f"{values[i, j]:.3f}", ha="center", va="center")

    plt.xticks(range(len(table.columns)), table.columns)
    plt.yticks(range(len(table.index)), table.index)
    plt.xlabel("gamma / gamma_ref")
    plt.ylabel("C")
    plt.title(title)
    plt.show()

In [ ]:
plot_grid_heatmap(svm_results, "train_f1_macro", "SVM RBF - train macro F1 (tuning subset)")
plot_grid_heatmap(svm_results, "val_f1_macro", "SVM RBF - validation macro F1")

## 7. Final model on the full training set

The best (C, gamma) of the grid is retrained on all 65 670 training beats. We keep the same absolute value of gamma found in the grid.

In [ ]:
best_index = svm_results["val_f1_macro"].idxmax()
C_best = svm_results.loc[best_index, "C"]
gamma_best = svm_results.loc[best_index, "gamma"]
print(f"Best on the subset: C = {C_best:g}, gamma = {gamma_best:.4g}, "
      f"val_f1_macro = {svm_results.loc[best_index, 'val_f1_macro']:.3f}")

svm_best = build_svm(C_best, gamma_best)

start = time.time()
svm_best.fit(X_train_svm, y_train)
print(f"Training time on the full set: {time.time() - start:.0f} s")

# Number of support vectors per class: a large fraction means a complex boundary
print("Support vectors per class:", svm_best.n_support_)

In [ ]:
y_train_pred_svm = svm_best.predict(X_train_svm)
y_val_pred_svm = svm_best.predict(X_val_svm)

scores_svm_train = evaluate(y_train, y_train_pred_svm)
scores_svm_val = evaluate(y_val, y_val_pred_svm)
print(f"train macro F1 = {scores_svm_train['f1_macro']:.3f}   val macro F1 = {scores_svm_val['f1_macro']:.3f}")

plot_confusion_matrix(y_val, y_val_pred_svm, title=f"SVM RBF (C = {C_best:g}) - validation")
print(classification_report(y_val, y_val_pred_svm, target_names=AAMI_NAMES, digits=3))

## 8. Test set

In [ ]:
y_test_pred_svm = svm_best.predict(X_test_svm)

plot_confusion_matrix(y_test, y_test_pred_svm, title=f"SVM RBF (C = {C_best:g}) - test")
print(classification_report(y_test, y_test_pred_svm, target_names=AAMI_NAMES, digits=3))

## 9. Conclusions

### Results

| Model | macro F1 | macro recall | train macro F1 |
|---|---|---|---|
| SVM RBF, validation | 0.939 | 0.933 | 0.982 |
| SVM RBF, test | 0.935 | 0.929 | |
| Best logistic regression (P1), validation | 0.554 | 0.834 | 0.552 |

Best hyperparameters: C = 10 and gamma = 3 * gamma_ref (0.0232).

### Grid search
- With small C and small gamma the validation macro F1 is 0.54, the same as logistic regression. This makes sense: with a small gamma the RBF kernel behaves almost like a linear model.
- With gamma = 10 * gamma_ref and C >= 1 the train F1 goes to 1 while the validation F1 drops to about 0.865, so the model overfits.
- The best value is inside the grid and not on the border, so the grid was wide enough.
- The model trained on the full training set does better than the one trained on the 20% subset (0.939 vs 0.914).

### Comparison with logistic regression
- Macro F1 goes from 0.554 to 0.939.
- The biggest difference is on the N class. Logistic regression had a recall of 0.73 on N, so about 27% of the normal beats were flagged as arrhythmias. The SVM has 0.996.
- For this reason the F1 of the rare classes improves a lot: S from 0.33 to 0.89, F from 0.18 to 0.85. The recall of F is a bit lower (0.845 vs 0.876), but logistic regression got it by predicting F for many beats that were not F.

### Overfitting
The gap between train (0.982) and validation (0.939) is small, and the test score (0.935) is almost the same as validation. The model generalises well and the hyperparameters were not overfitted to the validation set.

### Weak points
F is still the hardest class (F1 0.847 on validation, 0.820 on test), followed by S (0.888).